In [ ]:
# 상수 열, 정규화 열, 결측치 90% 이상 열, 완전 중복 행 삭제
# 객체 별 타임라인으로 정렬
# ST4000DM000_raw.parquet -> ST4000DM000_v1.parquet

import duckdb
import time

# 경로 설정
input_p = "C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_raw.parquet"
output_p = "C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v1.parquet"

print("🚀 전처리 파이프라인 가동: [열 정제(상수, 정규화, 결측률 90%) -> 고장 이후 유령 분리 -> 전체 중복행 제거 -> 다중 시계열 정렬]")
start_t = time.time()

con = duckdb.connect()

try:
    # 1. 전체 컬럼 정보 분석
    all_cols_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{input_p}')").fetchdf()
    all_cols = all_cols_df['column_name'].tolist()
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{input_p}')").fetchone()[0]

    # 2. 컬럼별 통계 전수조사 (상수/비어있는 열/결측률 판별용)
    print("-> 데이터 스캔 및 열 상태 분석 중...")
    # MIN, MAX를 통해 분산이 0인(모든 값이 동일한) 상수 열을 판별합니다.
    stats_sql = ", ".join([f"COUNT(\"{c}\") as \"{c}_cnt\", MIN(\"{c}\") as \"{c}_min\", MAX(\"{c}\") as \"{c}_max\"" for c in all_cols])
    stats_res = con.execute(f"SELECT {stats_sql} FROM read_parquet('{input_p}')").fetchdf().iloc[0]

    # 3. 제거 리스트 확정
    forced_drop = ['model', 'capacity_bytes']
    drop_list = []

    for col in all_cols:
        # A. 강제 제거(모델, 용량) 및 정규화(normalized) 열 우선 판단
        if col in forced_drop or col.endswith('_normalized'):
            drop_list.append(col)
            continue
        
        # B. 결측률 90% 이상 혹은 완전히 비어있는 열 판단
        non_null_cnt = stats_res[f"{col}_cnt"]
        missing_ratio = (total_rows - non_null_cnt) / total_rows

        if missing_ratio >= 0.9 or non_null_cnt == 0:
            drop_list.append(col)
            continue

        # C. 상수 열 제거 (분산 0 즉, 최솟값 == 최댓값)
        # 단, date나 serial_number와 같은 고유 식별자는 상수로 인식되지 않도록 예외 처리합니다.
        if col not in ['date', 'serial_number'] and stats_res[f"{col}_min"] == stats_res[f"{col}_max"]:
            drop_list.append(col)

    # 유효 컬럼에 테이블 별칭(t.) 추가
    valid_cols_sql = ", ".join([f"t.\"{col}\"" for col in all_cols if col not in drop_list])
    
    print(f"   * 제거된 열: {len(drop_list)}개")
    print(f"     (대상: 지정 제거, 정규화, 결측치 90%↑, 분산 0인 상수 열)")
    print(f"   * 유지된 열: {len(all_cols) - len(drop_list)}개")

    # 4. 핵심 정제 쿼리 실행
    # - DISTINCT: "모든 컬럼"의 데이터가 완전히 동일한 중복행만 1회성으로 제거합니다.
    # - FirstFailure & LEFT JOIN: 고장일(failure=1) 이후에 추가 기록된 유령 데이터를 절단합니다.
    # - ORDER BY: 다중 윈도우(패널 데이터) 학습을 위해 개체(serial_number) 및 날짜별(date)로 완벽하게 묶어 타임라인으로 정렬합니다.
    print("-> 고장 후 데이터 절단, 완전 중복행 제거 및 개체별 시계열(다중 윈도우) 정렬 중...")
    refine_sql = f"""
        COPY (
            WITH FirstFailure AS (
                SELECT serial_number, MIN(date) as fail_date
                FROM read_parquet('{input_p}')
                WHERE failure = 1
                GROUP BY serial_number
            )
            SELECT DISTINCT {valid_cols_sql}
            FROM read_parquet('{input_p}') t
            LEFT JOIN FirstFailure f ON t.serial_number = f.serial_number
            WHERE f.fail_date IS NULL OR t.date <= f.fail_date
            ORDER BY t.serial_number, t.date
        ) TO '{output_p}' (FORMAT PARQUET);
    """
    con.execute(refine_sql)

    print(f"\n✨ 전처리 완료! (소요 시간: {time.time() - start_t:.2f}초)")
    print(f"📦 저장 위치: {output_p}")

except Exception as e:
    print(f"❌ 파이프라인 중단 오류: {e}")
finally:
    con.close()

🚀 전처리 파이프라인 가동: [열 정제(상수, 정규화, 결측률 90%) -> 고장 이후 유령 분리 -> 전체 중복행 제거 -> 다중 시계열 정렬]
-> 데이터 스캔 및 열 상태 분석 중...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   * 제거된 열: 170개
     (대상: 지정 제거, 정규화, 결측치 90%↑, 분산 0인 상수 열)
   * 유지된 열: 27개
-> 고장 후 데이터 절단, 완전 중복행 제거 및 개체별 시계열(다중 윈도우) 정렬 중...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✨ 전처리 완료! (소요 시간: 228.44초)
📦 저장 위치: C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_temp.parquet


In [ ]:
# 중복 날짜가 있는지 확인

import duckdb
import pandas as pd

# 검사할 파일 경로 지정 (전처리 결과물)
check_p = "C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v1.parquet"

print("🔍 개체별 중복 날짜(Timeline 겹침) 검증 중...")

con = duckdb.connect()

try:
    # serial_number와 date를 그룹화하여 갯수가 2개 이상인 행을 찾습니다.
    duplicate_check_sql = f"""
        SELECT serial_number, date, COUNT(*) as log_count
        FROM read_parquet('{check_p}')
        GROUP BY serial_number, date
        HAVING COUNT(*) > 1
        ORDER BY log_count DESC, serial_number, date
    """
    
    dup_df = con.execute(duplicate_check_sql).fetchdf()
    
    if len(dup_df) == 0:
        print("✅ 완벽합니다! 모든 개체에 대해 중복된 날짜가 존재하지 않는 클린 타임라인입니다.")
    else:
        print(f"⚠️ 경고: {len(dup_df)}건의 (serial_number, date) 조합에서 중복된 날짜값이 발견되었습니다!")
        # 상위 5개의 중복 샘플 출력
        print("\n[상위 5건 중복 샘플]")
        display(dup_df.head(5))
        
        # 전체 중복 건수가 많을 수 있으니 총 중복 로그 개수도 요약해서 보여줍니다.
        total_dup_logs = dup_df['log_count'].sum() - len(dup_df) # 정상 1건 제외한 진짜 '중복' 건수
        print(f"\n총 잉여 중복 라인 수: {total_dup_logs}개")

except Exception as e:
    print(f"❌ 검증 중 오류 발생: {e}")
finally:
    con.close()


🔍 개체별 중복 날짜(Timeline 겹침) 검증 중...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ 완벽합니다! 모든 개체에 대해 중복된 날짜가 존재하지 않는 클린 타임라인입니다.


In [1]:
# 1. 마지막이 1로 끝났다가 다시 기록되는 경우
#     → 해당 개체 삭제
# 2. 마지막이 0으로 끝났다가 다시 기록되는 경우 
#     → 1일 공백은 보간, 2일 이상 공백부터는 공백 전과 공백 후를 다른 시리얼 넘버를 부여. 
# (공백 후의 데이터들에 대해 기존 시리얼넘버 + 1씩 붙여주는 네이밍 규칙)  
# 3. 중간에 마지막이 0으로 끝난 경우   
#     → 마지막 10일 삭제
# 4. 빈 시계열 행 생성
# 5. Foword fill 적용

import duckdb
import os
import time

# 입출력 및 임시 처리용 데이터 디렉토리
data_dir = "C:/Workspace/06_ML_projdect/26_1_COIN/data"
input_p = f"{data_dir}/ST4000DM000_v1.parquet"
output_p = f"{data_dir}/ST4000DM000_v2.parquet"
db_path = f"{data_dir}/preprocess_80M_7500F.duckdb" # RAM 초과분을 버퍼링할 물리적 캐시 DB

print("🚀 7500F & 36GB RAM 워크스테이션 맞춤형 8천만행 파이프라인 가동")
start_t = time.time()

# 파일 기반 DB 연결 (In-Memory 폭파 방지)
if os.path.exists(db_path):
    os.remove(db_path)
    
con = duckdb.connect(db_path)

try:
    # --- [🔥 7500F & 36GB 스펙 맞춤 하드 튜닝] ---
    # 시스템 메모리 36GB 중 안전하게 28GB 할당, OS/브라우저 여유 공간 8GB 보호
    con.execute("PRAGMA memory_limit='28GB'")
    # 라이젠 7500F의 12스레드를 극한으로 갈구어 병렬 처리 속도 극대화
    con.execute("PRAGMA threads=12") 
    con.execute(f"PRAGMA temp_directory='{data_dir}/duckdb_temp_80m'")
    
    # -------------------------------------------------------------
    # [STEP 1] RawData 로드 및 좀비(Ghost) 개체 파기 후 임시 물리 테이블 저장
    # -------------------------------------------------------------
    print("\n[1/4] 유령 개체 색출 및 클렌징 중...")
    step1_sql = f"""
        CREATE TABLE t1_clean AS 
        WITH GhostSerials AS (
            SELECT serial_number
            FROM read_parquet('{input_p}')
            GROUP BY serial_number
            HAVING MAX(CASE WHEN failure = 1 THEN CAST(date AS DATE) ELSE NULL END) < MAX(CAST(date AS DATE))
        )
        SELECT *
        FROM read_parquet('{input_p}')
        WHERE serial_number NOT IN (SELECT serial_number FROM GhostSerials);
    """
    con.execute(step1_sql)
    print(f"  -> STEP 1 완료 (소요: {time.time() - start_t:.2f}초)")

    # -------------------------------------------------------------
    # [STEP 2] 날짜 간극 누적합 계산 및 거대 시퀀스 식별자 쪼개기 (_1, _2)
    # -------------------------------------------------------------
    step_t = time.time()
    print("[2/4] 8천만 행 윈도우 정렬 및 다중 스레드 분리 연산 중...")
    step2_sql = """
        CREATE TABLE t2_chunked AS 
        WITH LaggedData AS (
            SELECT *,
                   LAG(date) OVER (PARTITION BY serial_number ORDER BY date) as prev_date
            FROM t1_clean
        ),
        GapFlagged AS (
            SELECT *,
                   CASE WHEN prev_date IS NOT NULL 
                         AND date_diff('day', CAST(prev_date AS DATE), CAST(date AS DATE)) >= 3 
                        THEN 1 ELSE 0 END as gap_flag
            FROM LaggedData
        ),
        ChunkedData AS (
            SELECT *,
                   SUM(gap_flag) OVER (PARTITION BY serial_number ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as chunk_id
            FROM GapFlagged
        )
        SELECT * EXCLUDE(prev_date, gap_flag),
               CASE WHEN chunk_id = 0 THEN serial_number 
                    ELSE serial_number || '_' || CAST(chunk_id AS VARCHAR) 
               END as new_serial
        FROM ChunkedData;
    """
    con.execute(step2_sql)
    print(f"  -> STEP 2 완료 (소요: {time.time() - step_t:.2f}초)")

    # -------------------------------------------------------------
    # [STEP 3] 라벨 오염 차단용 0으로 이탈한 개체의 마지막 10일치 잘라내기
    # -------------------------------------------------------------
    step_t = time.time()
    print("[3/4] 중도 이탈 개체의 마지막 10일 오염 데이터 절단 연산 중...")
    step3_sql = """
        CREATE TABLE t3_trimmed AS 
        WITH GlobalMax AS (
            SELECT MAX(CAST(date AS DATE)) as g_max_date FROM t2_chunked
        ),
        SeqEndings AS (
            SELECT new_serial, 
                   MAX(CAST(date AS DATE)) as seq_max_date,
                   MAX(failure) as seq_failure_status
            FROM t2_chunked
            GROUP BY new_serial
        ),
        ToDrop10Days AS (
            SELECT s.new_serial, s.seq_max_date
            FROM SeqEndings s
            CROSS JOIN GlobalMax g
            WHERE s.seq_failure_status = 0 
              AND s.seq_max_date < g.g_max_date
        )
        SELECT s.* EXCLUDE(chunk_id)
        FROM t2_chunked s
        LEFT JOIN ToDrop10Days d ON s.new_serial = d.new_serial
        WHERE d.new_serial IS NULL 
           OR CAST(s.date AS DATE) <= d.seq_max_date - INTERVAL 10 DAY;
    """
    con.execute(step3_sql)
    print(f"  -> STEP 3 완료 (소요: {time.time() - step_t:.2f}초)")

    # -------------------------------------------------------------
    # [STEP 4] 1일차 빈 달력 생성 및 전체 피처 병렬 보간
    # -------------------------------------------------------------
    step_t = time.time()
    print("[4/4] 1일 병합 슬롯 생성 및 피처 전체 보간(FFill) 후 파일 추출 중...")
    
    cols_df = con.execute("DESCRIBE SELECT * FROM t3_trimmed").fetchdf()
    feature_cols = [c for c in cols_df['column_name'].tolist() if c not in ['serial_number', 'new_serial', 'date', 'failure']]

    # 다중 스레드 환경에서 동작하는 동적 FFill
    ffill_sqls = []
    for col in feature_cols:
         ffill_sqls.append(f"LAST_VALUE(t.\"{col}\" IGNORE NULLS) OVER (PARTITION BY c.new_serial ORDER BY c.date) as \"{col}\"")
    ffill_str = ",\n                ".join(ffill_sqls)

    step4_sql = f"""
        COPY (
            WITH BoundaryDates AS (
                SELECT new_serial, MIN(CAST(date AS DATE)) as min_date, MAX(CAST(date AS DATE)) as max_date
                FROM t3_trimmed
                GROUP BY new_serial
            ),
            Calendar AS (
                SELECT new_serial, 
                       CAST(UNNEST(generate_series(min_date, max_date, INTERVAL 1 DAY)) AS DATE) as date
                FROM BoundaryDates
            )
            SELECT 
                c.new_serial as serial_number,
                c.date,
                COALESCE(t.failure, 0) as failure,
                {ffill_str}
            FROM Calendar c
            LEFT JOIN t3_trimmed t 
              ON c.new_serial = t.new_serial 
             AND c.date = CAST(t.date AS DATE)
            ORDER BY c.new_serial, c.date
        ) TO '{output_p}' (FORMAT PARQUET);
    """
    con.execute(step4_sql)
    print(f"  -> STEP 4 완료 (소요: {time.time() - step_t:.2f}초)")
    
    print(f"\n🎉 모든 전처리 완수! (총 소요 시간: {time.time() - start_t:.2f}초)")
    print(f"📦 변환된 텐서 시계열 저장됨: {output_p}")

except Exception as e:
    print(f"❌ 앗, 파이프라인 구동 중 오류가 났습니다: {e}")
finally:
    con.close()
    
    # 성공적으로 작업 끝마친 후 캐시 DB 정리
    if os.path.exists(db_path):
        os.remove(db_path)


🚀 7500F & 36GB RAM 워크스테이션 맞춤형 8천만행 파이프라인 가동

[1/4] 유령 개체 색출 및 클렌징 중...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> STEP 1 완료 (소요: 139.45초)
[2/4] 8천만 행 윈도우 정렬 및 다중 스레드 분리 연산 중...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> STEP 2 완료 (소요: 546.24초)
[3/4] 중도 이탈 개체의 마지막 10일 오염 데이터 절단 연산 중...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> STEP 3 완료 (소요: 160.11초)
[4/4] 1일 병합 슬롯 생성 및 피처 전체 보간(FFill) 후 파일 추출 중...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> STEP 4 완료 (소요: 1022.31초)

🎉 모든 전처리 완수! (총 소요 시간: 1868.11초)
📦 변환된 텐서 시계열 저장됨: C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v2.parquet


In [5]:
# - 타겟변수 레이블링
#     - 고장으로부터 10일 내의 구간`D-1 ~ D-10`을 모두 1(고장 임박)로 레이블링
#     - 고장 당일 `D-DAY` 는 삭제 (오늘 고장나는지를 예측하는 것이 아니기 때문)
# 1	Raw Read Error Rate	48비트 Hex 패킹. 상위 16비트 오류 수, 하위 32비트 총 읽기 수.	Read_Error_Count, Total_Reads
# 7	Seek Error Rate	48비트 Hex 패킹. 헤드 탐색 메커니즘의 오류와 전체 탐색 수 분리.	seek_error_count, Total_Seeks

import duckdb

con = duckdb.connect()

print("데이터 변환 및 저장을 시작합니다...")

# [추가됨]: 전체 컬럼 목록을 불러와 나머지 smart_ 변수들을 모두 BIGINT로 바꿔주는 구문 자동 생성
columns_df = con.execute("DESCRIBE SELECT * FROM '../data/ST4000DM000_v2.parquet'").df()
smart_cols = [c for c in columns_df['column_name'] if c.startswith('smart_') and c not in ['smart_1_raw', 'smart_7_raw']]
cast_queries = ",\n        ".join([f"CAST({col} AS BIGINT) AS {col}" for col in smart_cols])

query = f"""
COPY (
    WITH prep AS (
        SELECT 
            *,
            MIN(CASE WHEN failure = 1 THEN date ELSE NULL END) OVER (PARTITION BY serial_number) AS fail_date
        FROM '../data/ST4000DM000_v2.parquet'
    )
    SELECT 
        -- [수정됨]: 기존 * EXCLUDE 대신, 고유값/날짜를 따로 빼고 나머지 컬럼은 모두 숫자(BIGINT)로 변환해 가져옵니다.
        serial_number,
        date,
        {cast_queries},
        
        -- [수정됨]: Read_Error_Count는 분산이 0이므로 삭제하고 Total_Reads만 남김 (언더바 유지)
        (CAST(smart_1_raw AS BIGINT) & 4294967295) AS Total_Reads,
        
        -- Seek 에러 및 전체 탐색 수 분리 (언더바 유지)
        (CAST(smart_7_raw AS BIGINT) >> 32) AS seek_error_count,
        (CAST(smart_7_raw AS BIGINT) & 4294967295) AS Total_Seeks,
        
        -- 타겟 변수 레이블링
        CAST(
            CASE 
                WHEN fail_date IS NOT NULL AND date >= fail_date - INTERVAL 10 DAYS THEN 1 
                ELSE 0 
            END 
        AS BIGINT) AS failure

    FROM prep
    WHERE fail_date IS NULL OR date < fail_date
    
) TO '../data/ST4000DM000_v3.parquet' (FORMAT PARQUET);
"""

con.execute(query)

print("작업 완료! '../data/ST4000DM000_v3.parquet' 파일로 저장되었습니다.")


데이터 변환 및 저장을 시작합니다...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

작업 완료! '../data/ST4000DM000_v3.parquet' 파일로 저장되었습니다.


In [2]:
# smart_188_raw 디코딩

import duckdb
import time

input_file = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet"
output_file = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet"

print(f"🚀 {input_file} 읽기 및 전처리 시작...")
start_time = time.time()

# 188번을 디코딩하고 바로 V3 파일로 저장하는 쿼리
query = f"""
COPY (
    SELECT 
        * EXCLUDE (smart_188_raw), -- 기존 노이즈 컬럼 제거
        
        -- 188번 비트 연산 (결측치 등 에러 방지를 위해 BIGINT 강제 캐스팅)
        (smart_188_raw::BIGINT & 65535) AS timeout_total,           
        ((smart_188_raw::BIGINT >> 16) & 65535) AS timeout_5s,
        ((smart_188_raw::BIGINT >> 32) & 65535) AS Timeout_7s
        
    FROM '{input_file}'
) TO '{output_file}' (FORMAT PARQUET);
"""

# 쿼리 실행 (이미 v3 파일이 존재하면 자동으로 덮어씌웁니다)
duckdb.sql(query)

end_time = time.time()
print(f"✅ 전처리 완료! 결과물이 '{output_file}'로 성공적으로 덮어씌워졌습니다. (소요시간: {end_time - start_time:.2f}초)")

🚀 C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet 읽기 및 전처리 시작...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ 전처리 완료! 결과물이 'C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet'로 성공적으로 덮어씌워졌습니다. (소요시간: 8.97초)


In [3]:
# smart_190_raw, smart_194_raw 100도 이상인 값은 Foward fill로 덮어씌우기

import duckdb
import os
import time

# 원본 파일 경로 및 임시 파일 경로 설정
target_file = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet"
temp_file = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3_temp.parquet"

print(f"🚀 '{target_file}' 온도 이상치(100도 이상) 전처리 시작...")
start_time = time.time()

# 1. 100도 이상 NULL 처리 후 윈도우 함수로 이전 값 덮어쓰기 (임시 파일로 저장)
query = f"""
COPY (
    SELECT 
        * EXCLUDE (smart_190_raw, smart_194_raw),
        
        -- 현재 디스크(serial_number)의 날짜(date) 순으로 정렬 후 빈칸을 이전 온도로 채움
        LAST_VALUE(CASE WHEN smart_190_raw >= 100 THEN NULL ELSE smart_190_raw END IGNORE NULLS) 
            OVER (PARTITION BY serial_number ORDER BY date 
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS smart_190_raw,
                  
        LAST_VALUE(CASE WHEN smart_194_raw >= 100 THEN NULL ELSE smart_194_raw END IGNORE NULLS) 
            OVER (PARTITION BY serial_number ORDER BY date 
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS smart_194_raw
                  
    FROM '{target_file}'
) TO '{temp_file}' (FORMAT PARQUET);
"""

# 쿼리 실행
duckdb.sql(query)

# 2. 안전한 덮어쓰기 (기존 원본을 덮어씌움)
if os.path.exists(temp_file):
    os.replace(temp_file, target_file)
    
end_time = time.time()
print(f"✅ V3 파일 덮어쓰기 완료! (총 소요 시간: {end_time - start_time:.2f}초)")

# 검증 (최댓값이 100 아래로 잘 잡혔는지 확인)
check_query = f"SELECT MAX(smart_190_raw), MAX(smart_194_raw) FROM '{target_file}'"
print("🔍 덮어쓰기 완료 후 온도 최댓값 확인:", duckdb.query(check_query).fetchall())


🚀 'C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet' 온도 이상치(100도 이상) 전처리 시작...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ V3 파일 덮어쓰기 완료! (총 소요 시간: 77.32초)
🔍 덮어쓰기 완료 후 온도 최댓값 확인: [(97, 97)]


In [1]:
import duckdb
import os

src = r'C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet'
tmp = r'C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3_tmp.parquet'

con = duckdb.connect()

con.execute(f"""
    COPY (
        SELECT * EXCLUDE (Timeout_7_5s)
        FROM read_parquet('{src}')
    )
    TO '{tmp}'
    (FORMAT PARQUET)
""")

con.close()

os.replace(tmp, src)
print("완료: smart_12_raw, smart_240_raw, Timeout_7_5s 컬럼 삭제됨")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

완료: smart_12_raw, smart_240_raw, Timeout_7_5s 컬럼 삭제됨
